# 📊 HDFC Bank — 6-Checkpoint Trade Checklist
## Senior Data Scientist | TA-Lib Edition | Weekly Data | 2 Years

| # | Checkpoint | yfinance Data | Library |
|---|---|---|---|
| CP1 | Candlestick Pattern Detection | Open, High, Low, Close | **TA-Lib** |
| CP2 | Support & Resistance + Stop-Loss | High, Low, Close | Custom/pandas |
| CP3 | Volume > 10-Week Average | Volume | pandas rolling |
| CP4 | Dow Theory Trend & Phase | Close | **TA-Lib** SMA |
| CP5 | RSI + MACD Confirmation + Position Sizing | Close | **TA-Lib** |
| CP6 | Reward-to-Risk Ratio (RRR ≥ 1.5) | S/R, Entry, SL | Custom math |


## ⚙️ Step 0 — Install Dependencies

In [ ]:
import subprocess
for pkg in ['yfinance','TA-Lib','plotly','kaleido','nbformat']:
    r = subprocess.run(['pip','install','-q',pkg], capture_output=True, text=True)
    print('✅' if r.returncode==0 else '❌', pkg)
print('Done.')

## 📦 Step 1 — Imports

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import talib
import plotly.graph_objects as go
import plotly.io as pio
import json, warnings, os
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
pio.templates.default = 'plotly_dark'
print(f'✅ TA-Lib {talib.__version__} | pandas {pd.__version__}')

## 📥 Step 2 — Fetch 2-Year Weekly Data

In [ ]:
TICKER   = 'HDFCBANK.NS'
PERIOD   = '2y'
INTERVAL = '1wk'

raw = yf.download(TICKER, period=PERIOD, interval=INTERVAL, auto_adjust=True)
raw.columns = raw.columns.droplevel(1)
raw.dropna(inplace=True)
df = raw.copy()

print(f'✅ {len(df)} weekly bars | {df.index[0].date()} → {df.index[-1].date()}')
print(f'   Latest Close: ₹{df["Close"].iloc[-1]:.2f}')
df.tail(5)

## 🔢 Step 3 — Compute Technical Indicators (TA-Lib)

TA-Lib requires **float64 numpy arrays**. We compute RSI, MACD, SMA, Bollinger Bands, and ATR.

In [ ]:
# Convert OHLCV to float64 numpy (TA-Lib requirement)
op  = df['Open'].values.astype(float)
hi  = df['High'].values.astype(float)
lo  = df['Low'].values.astype(float)
cl  = df['Close'].values.astype(float)
vol = df['Volume'].values.astype(float)

# RSI (14-period)
df['RSI'] = talib.RSI(cl, timeperiod=14)

# MACD (12, 26, 9)
df['MACD'], df['MACD_Sig'], df['MACD_Hist'] = talib.MACD(cl, fastperiod=12, slowperiod=26, signalperiod=9)

# SMAs — on weekly: SMA10≈50d, SMA20≈100d, SMA40≈200d
df['SMA10'] = talib.SMA(cl, timeperiod=10)
df['SMA20'] = talib.SMA(cl, timeperiod=20)
df['SMA40'] = talib.SMA(cl, timeperiod=40)
df['EMA20'] = talib.EMA(cl, timeperiod=20)

# Bollinger Bands (20, 2σ)
df['BB_U'], df['BB_M'], df['BB_L'] = talib.BBANDS(cl, timeperiod=20, nbdevup=2, nbdevdn=2, matype=0)

# ATR — measures weekly volatility range
df['ATR'] = talib.ATR(hi, lo, cl, timeperiod=14)

# Volume 10-week rolling average
df['Vol10MA'] = df['Volume'].rolling(10).mean()

# Drop NaN warmup rows
df.dropna(subset=['RSI','MACD','SMA40','Vol10MA'], inplace=True)

# Refresh numpy arrays
op  = df['Open'].values.astype(float)
hi  = df['High'].values.astype(float)
lo  = df['Low'].values.astype(float)
cl  = df['Close'].values.astype(float)
vol = df['Volume'].values.astype(float)

print(f'✅ Indicators computed | {len(df)} rows')
df[['Close','RSI','MACD','MACD_Sig','SMA10','SMA40','BB_U','BB_L','ATR']].tail(5)

## 🕯️ CP1 — Candlestick Pattern Detection (TA-Lib)

TA-Lib provides 61 CDL functions. Output: `+100` = Bullish, `-100` = Bearish, `0` = None.

In [ ]:
# ── Bullish Patterns ────────────────────────────────────
df['HAMMER']       = talib.CDLHAMMER(op, hi, lo, cl)
df['INV_HAMMER']   = talib.CDLINVERTEDHAMMER(op, hi, lo, cl)
df['BULL_ENGULF']  = talib.CDLENGULFING(op, hi, lo, cl).clip(lower=0)      # keep +100
df['MORNING_STAR'] = talib.CDLMORNINGSTAR(op, hi, lo, cl, penetration=0)
df['PIERCING']     = talib.CDLPIERCING(op, hi, lo, cl)
df['THREE_WHITE']  = talib.CDL3WHITESOLDIERS(op, hi, lo, cl)
df['DRAGONFLY']    = talib.CDLDRAGONFLYDOJI(op, hi, lo, cl)
df['MARUBOZU_B']   = talib.CDLMARUBOZU(op, hi, lo, cl).clip(lower=0)

# ── Bearish Patterns ────────────────────────────────────
df['HANGING_MAN']  = talib.CDLHANGINGMAN(op, hi, lo, cl)
df['SHOOT_STAR']   = talib.CDLSHOOTINGSTAR(op, hi, lo, cl)
df['BEAR_ENGULF']  = talib.CDLENGULFING(op, hi, lo, cl).clip(upper=0)      # keep -100
df['EVENING_STAR'] = talib.CDLEVENINGSTAR(op, hi, lo, cl, penetration=0)
df['DARK_CLOUD']   = talib.CDLDARKCLOUDCOVER(op, hi, lo, cl, penetration=0.5)
df['THREE_BLACK']  = talib.CDL3BLACKCROWS(op, hi, lo, cl)
df['GRAVESTONE']   = talib.CDLGRAVESTONEDOJI(op, hi, lo, cl)
df['MARUBOZU_BK']  = talib.CDLMARUBOZU(op, hi, lo, cl).clip(upper=0)

# ── Neutral / Reversal ───────────────────────────────────
df['DOJI']         = talib.CDLDOJI(op, hi, lo, cl)
df['SPINNING_TOP'] = talib.CDLSPINNINGTOP(op, hi, lo, cl)
df['HARAMI']       = talib.CDLHARAMI(op, hi, lo, cl)
df['RICKSHAW']     = talib.CDLRICKSHAWMAN(op, hi, lo, cl)

BULL_COLS = ['HAMMER','INV_HAMMER','BULL_ENGULF','MORNING_STAR','PIERCING','THREE_WHITE','DRAGONFLY','MARUBOZU_B']
BEAR_COLS = ['HANGING_MAN','SHOOT_STAR','BEAR_ENGULF','EVENING_STAR','DARK_CLOUD','THREE_BLACK','GRAVESTONE','MARUBOZU_BK']
NEUT_COLS = ['DOJI','SPINNING_TOP','HARAMI','RICKSHAW']

def pattern_label(row):
    found = []
    for c in BULL_COLS:
        if row[c] == 100:  found.append(f'🟢 {c}')
    for c in BEAR_COLS:
        if row[c] == -100: found.append(f'🔴 {c}')
    for c in NEUT_COLS:
        if row[c] != 0:    found.append(f'🟡 {c}')
    return ', '.join(found) if found else '—'

df['PATTERN'] = df.apply(pattern_label, axis=1)

latest_pattern = df['PATTERN'].iloc[-1]
c1 = (latest_pattern != '—')

print(f'✅ CP1 — Candlestick Pattern (TA-Lib)')
print(f'   Latest week   : {df.index[-1].date()}')
print(f'   Pattern(s)    : {latest_pattern if c1 else "❌ No pattern on latest bar"}')
print(f'   CP1 Status    : {"✅ PASS" if c1 else "❌ FAIL"}')
print(f'\n   Patterns in last 15 weeks:')
print(df[df['PATTERN']!='—'].tail(15)[['Close','PATTERN']].to_string())

## 📐 CP2 — Support & Resistance + Stop-Loss

**Fractal pivot method:** A bar's Low is support if it's the lowest in a ±5 bar window (~2.5 months each side on weekly data). Stop-Loss = 1% below nearest support.

In [ ]:
def compute_sr_levels(df, window=5):
    support, resistance = [], []
    n = len(df)
    for i in range(window, n - window):
        if df['Low'].iloc[i]  == df['Low'].iloc[i-window:i+window+1].min():
            support.append((df.index[i], round(float(df['Low'].iloc[i]), 2)))
        if df['High'].iloc[i] == df['High'].iloc[i-window:i+window+1].max():
            resistance.append((df.index[i], round(float(df['High'].iloc[i]), 2)))
    return support, resistance

support_pts, resist_pts = compute_sr_levels(df, window=5)

current_price      = round(float(df['Close'].iloc[-1]), 2)
sup_below          = sorted([p for _, p in support_pts if p < current_price])
res_above          = sorted([p for _, p in resist_pts  if p > current_price])
nearest_support    = sup_below[-1] if sup_below else round(current_price * 0.95, 2)
nearest_resistance = res_above[0]  if res_above else round(current_price * 1.05, 2)
stop_loss          = round(nearest_support * 0.99, 2)
c2 = len(sup_below) > 0 and len(res_above) > 0

print(f'✅ CP2 — Support & Resistance')
print(f'   Current Price      : ₹{current_price}')
print(f'   Nearest Support    : ₹{nearest_support}  ({len(sup_below)} pivot(s) below)')
print(f'   Stop-Loss (SL)     : ₹{stop_loss}  (1% below support)')
print(f'   Nearest Resistance : ₹{nearest_resistance}  ({len(res_above)} pivot(s) above)')
print(f'   CP2 Status         : {"✅ PASS" if c2 else "❌ FAIL"}')
print(f'\n   Support  pivots (₹): {sup_below[-5:]}')
print(f'   Resist   pivots (₹): {res_above[:5]}')

## 📊 CP3 — Volume vs 10-Week Average

In [ ]:
vol_today = float(df['Volume'].iloc[-1])
vol_10avg = float(df['Vol10MA'].iloc[-1])
vol_ratio = round(vol_today / vol_10avg, 2)
c3        = vol_ratio > 1.0

print(f'✅ CP3 — Volume Analysis')
print(f'   This Week Volume  : {vol_today:,.0f}')
print(f'   10-Week Avg Vol   : {vol_10avg:,.0f}')
print(f'   Volume Ratio      : {vol_ratio}x')
print(f'   CP3 Status        : {"✅ PASS" if c3 else "❌ FAIL"}')

## 📈 CP4 — Dow Theory Trend & Phase

| Phase | Signal | Meaning |
|---|---|---|
| Bull Ph1 | Accumulation | Smart money buys quietly |
| Bull Ph2 | Public Participation | Trend confirmed, public enters |
| Bull Ph3 | Distribution | Smart money exits |
| Bear Ph1 | Distribution | Topping/rolling over |
| Bear Ph2 | Public Panic | Rapid price decline |
| Bear Ph3 | Despair | Capitulation / potential bottom |

In [ ]:
latest    = df.iloc[-1]
sma10_val = round(float(latest['SMA10']), 2)
sma20_val = round(float(latest['SMA20']), 2)
sma40_val = round(float(latest['SMA40']), 2)
cp_val    = round(float(latest['Close']),  2)

dow_uptrend = sma10_val > sma40_val

recent8 = df.tail(8)
hh = float(recent8['High'].iloc[-1]) > float(recent8['High'].iloc[:-1].max())
hl = float(recent8['Low'].iloc[-1])  > float(recent8['Low'].iloc[:-1].min())
ll = float(recent8['Low'].iloc[-1])  < float(recent8['Low'].iloc[:-1].min())
lh = float(recent8['High'].iloc[-1]) < float(recent8['High'].iloc[:-1].max())

if dow_uptrend:
    if hh and hl:        phase = 'Bull Phase 2: Public Participation ✅ — Confirmed Uptrend'
    elif cp_val>sma20_val: phase = 'Bull Phase 1: Accumulation — Early Bull'
    else:                phase = 'Bull Phase 3: Distribution ⚠️ — Possible Reversal'
else:
    if ll and lh:        phase = 'Bear Phase 2: Public Panic 🔴 — Active Downtrend'
    elif cp_val<sma20_val: phase = 'Bear Phase 3: Despair — Oversold/Capitulation Zone'
    else:                phase = 'Bear Phase 1: Distribution — Rolling Over'

c4 = dow_uptrend

print(f'✅ CP4 — Dow Theory')
print(f'   SMA10: ₹{sma10_val}  SMA20: ₹{sma20_val}  SMA40(≈200d): ₹{sma40_val}')
print(f'   Primary Trend : {"🟢 UPTREND" if dow_uptrend else "🔴 DOWNTREND"}')
print(f'   HH={hh} | HL={hl} | LL={ll} | LH={lh}')
print(f'   Market Phase  : {phase}')
print(f'   CP4 Status    : {"✅ PASS" if c4 else "❌ FAIL"}')

## 🔬 CP5 — RSI + MACD Confirmation + Position Sizing

**Position Size Rule:** Both confirm → 100% | One confirms → 50% | Neither → 0% (skip)

In [ ]:
latest    = df.iloc[-1]
rsi_val   = round(float(latest['RSI']),       2)
macd_val  = round(float(latest['MACD']),      2)
macd_sig  = round(float(latest['MACD_Sig']),  2)
macd_his  = round(float(latest['MACD_Hist']), 2)
atr_val   = round(float(latest['ATR']),       2)

rsi_oversold   = rsi_val < 30
rsi_overbought = rsi_val > 70
rsi_bullish    = rsi_val < 60

macd_bullish   = macd_val > macd_sig
macd_momentum  = macd_his > 0
prev_hist      = float(df['MACD_Hist'].iloc[-2])
macd_growing   = macd_his > prev_hist

both_confirm = rsi_bullish and macd_bullish
one_confirm  = rsi_bullish or  macd_bullish
pos_size     = 1.0 if both_confirm else (0.5 if one_confirm else 0.0)
c5           = both_confirm

print(f'✅ CP5 — RSI + MACD')
print(f'   RSI (14w)       : {rsi_val}  {"⚠️  OVERSOLD" if rsi_oversold else ("⚠️  OVERBOUGHT" if rsi_overbought else "")}')
print(f'   RSI Bullish <60 : {"✅ Yes" if rsi_bullish else "❌ No"}')
print(f'   MACD / Signal   : {macd_val} / {macd_sig}  → {"✅ Bullish (MACD>Sig)" if macd_bullish else "❌ Bearish"}')
print(f'   MACD Histogram  : {macd_his}  ({"Positive ✅" if macd_momentum else "Negative ❌"})  Trend={"Growing" if macd_growing else "Shrinking"}')
print(f'   ATR (volatility): ₹{atr_val}/week')
print(f'   Position Size   : {pos_size*100:.0f}% of planned capital')
print(f'   CP5 Status      : {"✅ PASS" if c5 else ("⚠️  PARTIAL 50%" if one_confirm else "❌ FAIL")}')

## ⚖️ CP6 — Reward-to-Risk Ratio (RRR)

```
RRR = (Target − Entry) / (Entry − Stop Loss)
```

Rule: **RRR ≥ 1.5** to take the trade (ideally ≥ 2.0).

In [ ]:
entry_price = current_price
risk        = round(entry_price - stop_loss, 2)
reward      = round(nearest_resistance - entry_price, 2)
rrr         = round(reward / risk, 2) if risk > 0 else 0
c6          = rrr >= 1.5

RISK_BUDGET   = 10_000
shares_qty    = int(RISK_BUDGET / risk) if risk > 0 else 0
total_capital = round(shares_qty * entry_price, 2)
potential_pnl = round(shares_qty * reward, 2)

print(f'✅ CP6 — Reward-to-Risk Ratio')
print(f'   Entry Price      : ₹{entry_price}')
print(f'   Stop-Loss        : ₹{stop_loss}   Risk   = ₹{risk}')
print(f'   Target Price     : ₹{nearest_resistance}  Reward = ₹{reward}')
print(f'   RRR              : {rrr}x')
print(f'   CP6 Status       : {"✅ PASS (≥1.5)" if c6 else "❌ FAIL (<1.5)"}')
print(f'\n   Trade Sizing (₹{RISK_BUDGET:,} fixed risk):')
print(f'   Shares to Buy    : {shares_qty}')
print(f'   Capital Needed   : ₹{total_capital:,.0f}')
print(f'   Potential Profit : ₹{potential_pnl:,.0f}')
print(f'   Potential Loss   : ₹{RISK_BUDGET:,.0f} (capped)')

## 🏆 Final Scorecard

In [ ]:
checkpoints = {
    'CP1 Candlestick Pattern'   : (c1, latest_pattern if c1 else 'No pattern on latest bar'),
    'CP2 S&R Confirms Trade'    : (c2, f'SL=₹{stop_loss} | Target=₹{nearest_resistance}'),
    'CP3 Volume > 10-Wk Avg'    : (c3, f'{vol_ratio}x average'),
    'CP4 Dow Theory Uptrend'    : (c4, phase[:55]),
    'CP5 RSI+MACD Both Confirm' : (c5, f'RSI={rsi_val} | MACD {">" if macd_bullish else "<"} Signal'),
    'CP6 RRR >= 1.5'            : (c6, f'RRR = {rrr}x'),
}
score = sum(v for v,_ in checkpoints.values())
if   score>=5: verdict='🟢 HIGH CONFIDENCE — Consider entering trade'
elif score>=3: verdict='🟡 MODERATE — Trade with caution'
else:          verdict='🔴 LOW CONFIDENCE — DO NOT trade yet'

print('='*64)
print(f'  HDFC BANK WEEKLY CHECKLIST | {df.index[-1].date()} | ₹{current_price}')
print('='*64)
for name,(passed,detail) in checkpoints.items():
    print(f'  {"✅" if passed else "❌"}  {name:<32}  {detail}')
print('-'*64)
print(f'  SCORE  : {score}/6')
print(f'  VERDICT: {verdict}')
print('='*64)
print(f'  Entry ₹{entry_price} | SL ₹{stop_loss} | Target ₹{nearest_resistance} | RRR {rrr}x | Pos {pos_size*100:.0f}%')

## 📉 Chart 1 — Price, Bollinger Bands, S&R & TA-Lib Patterns (CP1+CP2)

In [ ]:
plot_df = df.tail(52).copy()
dates   = plot_df.index

fig1 = go.Figure()
fig1.add_trace(go.Candlestick(x=dates,open=plot_df['Open'],high=plot_df['High'],
    low=plot_df['Low'],close=plot_df['Close'],name='HDFCBANK',
    increasing_line_color='#26a69a',decreasing_line_color='#ef5350'))
fig1.add_trace(go.Scatter(x=dates,y=plot_df['BB_U'],name='BB Upper',
    line=dict(color='rgba(255,214,0,0.4)',dash='dot',width=1)))
fig1.add_trace(go.Scatter(x=dates,y=plot_df['BB_L'],name='BB Lower',
    line=dict(color='rgba(255,214,0,0.4)',dash='dot',width=1),
    fill='tonexty',fillcolor='rgba(255,214,0,0.05)'))
fig1.add_trace(go.Scatter(x=dates,y=plot_df['SMA10'],name='SMA10',line=dict(color='#42a5f5',width=2)))
fig1.add_trace(go.Scatter(x=dates,y=plot_df['SMA40'],name='SMA40(≈200d)',line=dict(color='#ab47bc',width=2)))
fig1.add_hline(y=nearest_support,    line_dash='dot', line_color='#66bb6a',annotation_text=f'Support ₹{nearest_support}',   annotation_position='bottom right')
fig1.add_hline(y=nearest_resistance, line_dash='dot', line_color='#ef5350',annotation_text=f'Resist ₹{nearest_resistance}',   annotation_position='top right')
fig1.add_hline(y=stop_loss,          line_dash='dash',line_color='#ff7043',annotation_text=f'SL ₹{stop_loss}',              annotation_position='bottom left')
for i in range(len(plot_df)):
    pat=plot_df['PATTERN'].iloc[i]
    if pat!='—':
        c='#26a69a' if '🟢' in pat else '#ef5350'
        fig1.add_annotation(x=dates[i],y=float(plot_df['High'].iloc[i])*1.008,
            text=pat.replace('🟢 ','').replace('🔴 ','')[:8],showarrow=False,
            font=dict(size=8,color=c),bgcolor='rgba(0,0,0,0.7)')
fig1.update_layout(title='HDFC Bank Weekly — Price + BB + S&R + TA-Lib Patterns',
    xaxis_rangeslider_visible=False,template='plotly_dark',
    legend=dict(orientation='h',yanchor='bottom',y=1.05,xanchor='center',x=0.5))
fig1.update_xaxes(title_text='Week')
fig1.update_yaxes(title_text='Price (₹)')
fig1.show()

## 📦 Chart 2 — Volume vs 10-Week Average (CP3)

In [ ]:
bar_colors=['#26a69a' if v>a else '#ef5350' for v,a in zip(plot_df['Volume'],plot_df['Vol10MA'])]
fig2=go.Figure()
fig2.add_trace(go.Bar(x=dates,y=plot_df['Volume']/1e6,name='Weekly Volume',marker_color=bar_colors))
fig2.add_trace(go.Scatter(x=dates,y=plot_df['Vol10MA']/1e6,name='10-Week Avg',line=dict(color='#ffa726',width=2.5)))
fig2.update_layout(title='HDFC Bank Weekly Volume vs 10-Week Average (CP3)',template='plotly_dark',
    legend=dict(orientation='h',yanchor='bottom',y=1.05,xanchor='center',x=0.5))
fig2.update_xaxes(title_text='Week')
fig2.update_yaxes(title_text='Volume (M shares)')
fig2.show()

## 📈 Chart 3 — Dow Theory Trend & Phase (CP4)

In [ ]:
fig3=go.Figure()
for i in range(1,len(plot_df)):
    bull=float(plot_df['SMA10'].iloc[i])>float(plot_df['SMA40'].iloc[i])
    fig3.add_vrect(x0=dates[i-1],x1=dates[i],
        fillcolor='rgba(38,166,154,0.08)' if bull else 'rgba(239,83,80,0.08)',line_width=0)
fig3.add_trace(go.Scatter(x=dates,y=plot_df['Close'],name='Close',line=dict(color='white',width=1.5)))
fig3.add_trace(go.Scatter(x=dates,y=plot_df['SMA10'],name='SMA10',line=dict(color='#42a5f5',width=2)))
fig3.add_trace(go.Scatter(x=dates,y=plot_df['SMA20'],name='SMA20',line=dict(color='#ffa726',width=1.5)))
fig3.add_trace(go.Scatter(x=dates,y=plot_df['SMA40'],name='SMA40(≈200d)',line=dict(color='#ab47bc',width=2.5)))
fig3.update_layout(title='HDFC Bank Dow Theory Weekly (CP4) — Green=Bull | Red=Bear',template='plotly_dark',
    legend=dict(orientation='h',yanchor='bottom',y=1.05,xanchor='center',x=0.5))
fig3.update_xaxes(title_text='Week')
fig3.update_yaxes(title_text='Price (₹)')
fig3.show()

## 🔬 Chart 4 — RSI & MACD (CP5)

In [ ]:
fig4=make_subplots(rows=2,cols=1,shared_xaxes=True,row_heights=[0.45,0.55],vertical_spacing=0.07,
    subplot_titles=['RSI (14-week)','MACD (12,26,9) Weekly'])
fig4.add_trace(go.Scatter(x=dates,y=plot_df['RSI'],name='RSI',line=dict(color='#42a5f5',width=2)),row=1,col=1)
fig4.add_hrect(y0=70,y1=100,fillcolor='rgba(239,83,80,0.10)',  line_width=0,row=1,col=1)
fig4.add_hrect(y0=0, y1=30, fillcolor='rgba(38,166,154,0.10)', line_width=0,row=1,col=1)
for lvl,clr in [(70,'#ef5350'),(50,'gray'),(30,'#66bb6a')]:
    fig4.add_hline(y=lvl,line_dash='dash',line_color=clr,line_width=1,row=1,col=1)
hist_clrs=['#26a69a' if v>=0 else '#ef5350' for v in plot_df['MACD_Hist']]
fig4.add_trace(go.Bar(x=dates,y=plot_df['MACD_Hist'],name='Histogram',marker_color=hist_clrs),row=2,col=1)
fig4.add_trace(go.Scatter(x=dates,y=plot_df['MACD'],    name='MACD',  line=dict(color='#42a5f5',width=2)),row=2,col=1)
fig4.add_trace(go.Scatter(x=dates,y=plot_df['MACD_Sig'],name='Signal',line=dict(color='#ffa726',width=2)),row=2,col=1)
fig4.update_layout(title='HDFC Bank Weekly RSI & MACD (CP5)',template='plotly_dark',
    legend=dict(orientation='h',yanchor='bottom',y=1.05,xanchor='center',x=0.5))
fig4.update_xaxes(title_text='Week',row=2,col=1)
fig4.update_yaxes(title_text='RSI', row=1,col=1)
fig4.update_yaxes(title_text='MACD',row=2,col=1)
fig4.show()

## 🏆 Chart 5 — Scorecard + RRR Waterfall (CP6)

In [ ]:
fig5=make_subplots(rows=1,cols=2,column_widths=[0.58,0.42],subplot_titles=['Checklist Scorecard','RRR Waterfall'])
cp_names=[k for k in checkpoints.keys()]
cp_res=[v for v,_ in checkpoints.values()]
sc_clrs=['#26a69a' if r else '#ef5350' for r in cp_res]
fig5.add_trace(go.Bar(y=cp_names,x=[1]*6,orientation='h',marker_color=sc_clrs,
    text=['✅ PASS' if r else '❌ FAIL' for r in cp_res],
    textposition='inside',textfont=dict(size=12,color='white')),row=1,col=1)
fig5.add_trace(go.Waterfall(orientation='v',
    measure=['absolute','relative','total','relative','total'],
    x=['Entry','− SL','Net Risk','+ Reward','Target'],
    y=[entry_price,-risk,0,reward,0],
    connector=dict(line=dict(color='gray',width=1)),
    decreasing=dict(marker_color='#ef5350'),
    increasing=dict(marker_color='#26a69a'),
    totals=dict(marker_color='#42a5f5'),
    text=[f'₹{abs(v):.0f}' for v in [entry_price,-risk,0,reward,0]],
    textposition='outside'),row=1,col=2)
vd='🟢 HIGH' if score>=5 else ('🟡 MODERATE' if score>=3 else '🔴 LOW')
fig5.update_layout(title=f'HDFC Bank Scorecard — {score}/6 | {vd} | RRR={rrr}x',
    template='plotly_dark',showlegend=False)
fig5.update_xaxes(title_text='Trade Levels',row=1,col=2)
fig5.update_yaxes(title_text='Checkpoint',  row=1,col=1)
fig5.update_yaxes(title_text='Price (₹)',   row=1,col=2)
fig5.show()

## 💾 Export Results to CSV & JSON

In [ ]:
import os
os.makedirs('output', exist_ok=True)
export_cols=['Open','High','Low','Close','Volume','RSI','MACD','MACD_Sig','MACD_Hist',
             'SMA10','SMA20','SMA40','BB_U','BB_M','BB_L','ATR','Vol10MA','PATTERN']
df[export_cols].to_csv('output/HDFCBANK_weekly_analysis.csv')
print('✅ output/HDFCBANK_weekly_analysis.csv')

summary={
    'Ticker':TICKER,'Date':str(df.index[-1].date()),
    'Price':current_price,'Score':f'{score}/6','Verdict':verdict,
    'CP1_Pattern':latest_pattern,
    'CP2_Support':nearest_support,'CP2_Resistance':nearest_resistance,'CP2_StopLoss':stop_loss,
    'CP3_VolumeRatio':vol_ratio,'CP4_Phase':phase,
    'CP5_RSI':rsi_val,'CP5_MACD':macd_val,'CP5_PositionSize':f'{pos_size*100:.0f}%',
    'CP6_Risk':risk,'CP6_Reward':reward,'CP6_RRR':rrr,
}
import json
with open('output/HDFCBANK_checklist_summary.json','w') as f:
    json.dump(summary,f,indent=2)
print('✅ output/HDFCBANK_checklist_summary.json')
print(json.dumps(summary,indent=2))